# Lightweight Spatially Adaptive Image Dehazing
### Real-Time / Edge Deployment Research

This notebook implements a complete training pipeline for a lightweight spatially-adaptive
encoder-decoder network designed for single image dehazing.

**Key research ideas:**
- Depthwise-separable convolutions for efficiency
- Spatial attention gating for non-uniform haze adaptation
- Multi-scale dilated bottleneck for multi-scale context
- Global residual learning
- Sub-million / low-million parameter target

---
## 1. Imports

In [ ]:
import os
import time
import random
import math
import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

import torchvision.transforms.functional as TF

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Configuration

In [ ]:
# ==================== CONFIGURATION ====================

# Paths
TRAIN_PATH = "/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/train"
TEST_PATH  = "/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/test"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

# Training hyperparameters
NUM_EPOCHS = 50
BATCH_SIZE = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Image dimensions
TRAIN_RESIZE = 286
CROP_SIZE = 256
TEST_RESIZE = 256

# Loss weights
CHARBONNIER_WEIGHT = 0.8
SSIM_WEIGHT = 0.2
CHARBONNIER_EPS = 1e-3

# Checkpointing
RESUME_TRAINING = True

# Reproducibility
SEED = 42

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Performance
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# Seed everything
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# Create checkpoint directory
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

---
## 3. Dataset and PairedTransform

In [ ]:
class PairedTransform:
    """
    Synchronized transforms for hazy-clean image pairs.
    Training: resize to 286x286, random 256x256 crop, random horizontal flip.
    Testing: resize to 256x256.
    """

    def __init__(self, train=True, resize=286, crop_size=256, test_resize=256):
        self.train = train
        self.resize = resize
        self.crop_size = crop_size
        self.test_resize = test_resize

    def __call__(self, hazy_img, clean_img):
        if self.train:
            # Resize both to resize x resize
            hazy_img = TF.resize(hazy_img, [self.resize, self.resize])
            clean_img = TF.resize(clean_img, [self.resize, self.resize])

            # Synchronized random crop
            i, j, h, w = self._get_random_crop_params(hazy_img, self.crop_size)
            hazy_img = TF.crop(hazy_img, i, j, h, w)
            clean_img = TF.crop(clean_img, i, j, h, w)

            # Synchronized random horizontal flip
            if random.random() > 0.5:
                hazy_img = TF.hflip(hazy_img)
                clean_img = TF.hflip(clean_img)
        else:
            # Test: just resize to test_resize x test_resize
            hazy_img = TF.resize(hazy_img, [self.test_resize, self.test_resize])
            clean_img = TF.resize(clean_img, [self.test_resize, self.test_resize])

        # Convert to tensor in [0,1]
        hazy_tensor = TF.to_tensor(hazy_img)
        clean_tensor = TF.to_tensor(clean_img)

        return hazy_tensor, clean_tensor

    @staticmethod
    def _get_random_crop_params(img, crop_size):
        """Get parameters for a random crop."""
        w, h = img.size  # PIL: (width, height)
        th = tw = crop_size
        if w == tw and h == th:
            return 0, 0, th, tw
        i = random.randint(0, h - th)
        j = random.randint(0, w - tw)
        return i, j, th, tw


class RESIDEDataset(Dataset):
    """
    RESIDE-6K paired hazy/clean dataset.
    Directory structure:
        root/
            GT/      (ground truth clean images)
            hazy/    (hazy images)
    """

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.hazy_dir = os.path.join(root_dir, "hazy")
        self.gt_dir = os.path.join(root_dir, "GT")

        # Get sorted list of filenames
        self.hazy_images = sorted(os.listdir(self.hazy_dir))
        self.gt_images = sorted(os.listdir(self.gt_dir))

        assert len(self.hazy_images) == len(self.gt_images), \
            f"Mismatch: {len(self.hazy_images)} hazy vs {len(self.gt_images)} GT images"

        print(f"Loaded {len(self.hazy_images)} image pairs from {root_dir}")

    def __len__(self):
        return len(self.hazy_images)

    def __getitem__(self, idx):
        hazy_path = os.path.join(self.hazy_dir, self.hazy_images[idx])
        gt_path = os.path.join(self.gt_dir, self.gt_images[idx])

        hazy_img = Image.open(hazy_path).convert("RGB")
        clean_img = Image.open(gt_path).convert("RGB")

        if self.transform:
            hazy_img, clean_img = self.transform(hazy_img, clean_img)

        return hazy_img, clean_img

---
## 4. DataLoaders

In [ ]:
# Create transforms
train_transform = PairedTransform(
    train=True,
    resize=TRAIN_RESIZE,
    crop_size=CROP_SIZE,
    test_resize=TEST_RESIZE
)

test_transform = PairedTransform(
    train=False,
    resize=TRAIN_RESIZE,
    crop_size=CROP_SIZE,
    test_resize=TEST_RESIZE
)

# Create datasets
train_dataset = RESIDEDataset(TRAIN_PATH, transform=train_transform)
test_dataset = RESIDEDataset(TEST_PATH, transform=test_transform)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

---
## 5. Model Components

Lightweight building blocks used throughout the architecture.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """
    Depthwise separable convolution: depthwise 3x3 + pointwise 1x1.
    Significantly fewer parameters and FLOPs than standard convolution.
    """

    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=1, dilation=1, bias=False):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            stride=stride, padding=padding, dilation=dilation,
            groups=in_channels, bias=bias
        )
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, bias=bias
        )

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class LightweightDownsample(nn.Module):
    """
    Downsample spatial resolution by 2x using strided depthwise-separable conv.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.down = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=2,
                      padding=1, groups=in_channels, bias=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.down(x)


class LightweightUpsample(nn.Module):
    """
    Upsample spatial resolution by 2x using bilinear interpolation + 1x1 conv.
    More efficient than transposed convolutions.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.conv(x)

---
## 6. Spatial Adaptive Block

The core building block: depthwise-separable convolution with spatial attention gating.
Allows the network to adapt its feature processing based on spatially varying haze density.

In [ ]:
class SpatialAttentionGate(nn.Module):
    """
    Lightweight spatial attention gate.
    Generates a spatial attention map via channel-squeeze and sigmoid gating.
    This enables the network to focus on regions with varying haze density.
    """

    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Sequential(
            # Reduce channels to a compact representation
            nn.Conv2d(channels, channels // 4, kernel_size=1, bias=False),
            nn.GELU(),
            # Depthwise conv to capture local spatial patterns
            nn.Conv2d(channels // 4, channels // 4, kernel_size=3, padding=1,
                      groups=channels // 4, bias=False),
            nn.GELU(),
            # Project back to full channels for gating
            nn.Conv2d(channels // 4, channels, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.gate(x)


class SpatialAdaptiveBlock(nn.Module):
    """
    Lightweight spatial-adaptive feature processing block.

    Architecture:
        Input
         ↓
        LayerNorm
         ↓
        Depthwise 3x3 convolution
         ↓
        Pointwise 1x1 convolution (expand)
         ↓
        GELU activation
         ↓
        Spatial attention gating
         ↓
        Pointwise 1x1 projection (compress)
         ↓
        Residual addition

    Uses depthwise-separable convolutions for efficiency.
    Spatial gating adapts processing to haze distribution.
    """

    def __init__(self, channels, expansion_ratio=2):
        super().__init__()
        expanded = channels * expansion_ratio

        # Channel-last LayerNorm via GroupNorm with 1 group
        self.norm = nn.GroupNorm(1, channels)

        # Depthwise 3x3
        self.dw_conv = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1,
            groups=channels, bias=False
        )

        # Pointwise expand
        self.pw_expand = nn.Conv2d(channels, expanded, kernel_size=1, bias=False)

        self.act = nn.GELU()

        # Spatial attention gating on expanded features
        self.spatial_gate = SpatialAttentionGate(expanded)

        # Pointwise compress back
        self.pw_compress = nn.Conv2d(expanded, channels, kernel_size=1, bias=False)

    def forward(self, x):
        residual = x

        out = self.norm(x)
        out = self.dw_conv(out)
        out = self.pw_expand(out)
        out = self.act(out)
        out = self.spatial_gate(out)
        out = self.pw_compress(out)

        return out + residual

---
## 7. Multi-Scale Bottleneck

Captures haze patterns at multiple spatial scales using parallel dilated depthwise
convolutions with different receptive fields. Computationally cheap alternative to
Transformer-based bottlenecks.

In [ ]:
class MultiScaleBottleneck(nn.Module):
    """
    Multi-scale context extraction at the bottleneck.

    Uses three parallel branches of dilated depthwise convolutions:
        - 3x3 dilation=1 (local context)
        - 3x3 dilation=2 (medium context)
        - 3x3 dilation=3 (wide context)

    Features are concatenated and fused with a 1x1 convolution.
    A residual connection preserves the input information.
    """

    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(1, channels)

        # Branch 1: local receptive field
        self.branch1 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1,
                      dilation=1, groups=channels, bias=False),
            nn.GELU()
        )

        # Branch 2: medium receptive field
        self.branch2 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=2,
                      dilation=2, groups=channels, bias=False),
            nn.GELU()
        )

        # Branch 3: wide receptive field
        self.branch3 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=3,
                      dilation=3, groups=channels, bias=False),
            nn.GELU()
        )

        # Fusion: 1x1 conv to merge 3x channels back to channels
        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 3, channels, kernel_size=1, bias=False),
            nn.GroupNorm(1, channels),
            nn.GELU()
        )

    def forward(self, x):
        normed = self.norm(x)

        b1 = self.branch1(normed)
        b2 = self.branch2(normed)
        b3 = self.branch3(normed)

        # Concatenate multi-scale features
        fused = self.fusion(torch.cat([b1, b2, b3], dim=1))

        return fused + x  # Residual

---
## 8. Encoder

Three-stage encoder with increasing channel dimensions (32 → 64 → 96).
Each stage uses 2 × SpatialAdaptiveBlock. Downsampling between stages.

In [ ]:
class EncoderStage(nn.Module):
    """Single encoder stage with N SpatialAdaptiveBlocks."""

    def __init__(self, channels, num_blocks=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[SpatialAdaptiveBlock(channels) for _ in range(num_blocks)]
        )

    def forward(self, x):
        return self.blocks(x)


class Encoder(nn.Module):
    """
    Three-stage encoder.
    Stage 1: 32 channels, 2 × SpatialAdaptiveBlock
    Stage 2: 64 channels, 2 × SpatialAdaptiveBlock
    Stage 3: 96 channels, 2 × SpatialAdaptiveBlock

    Returns features from each stage for skip connections.
    """

    def __init__(self, channels=(32, 64, 96)):
        super().__init__()
        c1, c2, c3 = channels

        self.stage1 = EncoderStage(c1, num_blocks=2)
        self.down1 = LightweightDownsample(c1, c2)

        self.stage2 = EncoderStage(c2, num_blocks=2)
        self.down2 = LightweightDownsample(c2, c3)

        self.stage3 = EncoderStage(c3, num_blocks=2)

    def forward(self, x):
        # Stage 1: 256x256 @ 32ch
        enc1 = self.stage1(x)

        # Downsample → 128x128 @ 64ch
        x = self.down1(enc1)

        # Stage 2: 128x128 @ 64ch
        enc2 = self.stage2(x)

        # Downsample → 64x64 @ 96ch
        x = self.down2(enc2)

        # Stage 3: 64x64 @ 96ch
        enc3 = self.stage3(x)

        return enc1, enc2, enc3

---
## 9. Decoder

Three-stage decoder mirroring the encoder (96 → 64 → 32).
Uses bilinear upsampling + 1x1 conv. Skip connections fused via 1x1 conv
to keep channel counts from growing excessively.

In [ ]:
class DecoderStage(nn.Module):
    """Single decoder stage with skip connection fusion and SpatialAdaptiveBlocks."""

    def __init__(self, in_channels, skip_channels, out_channels, num_blocks=2):
        super().__init__()
        # Fuse upsampled features + skip connection
        self.skip_fusion = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=1, bias=False),
            nn.GroupNorm(1, out_channels),
            nn.GELU()
        )
        self.blocks = nn.Sequential(
            *[SpatialAdaptiveBlock(out_channels) for _ in range(num_blocks)]
        )

    def forward(self, x, skip):
        # Concatenate upsampled features with skip connection
        x = torch.cat([x, skip], dim=1)
        x = self.skip_fusion(x)
        x = self.blocks(x)
        return x


class Decoder(nn.Module):
    """
    Three-stage decoder.
    Stage 3: 96ch → upsample → fuse with enc2 (64ch) → 64ch output
    Stage 2: 64ch → upsample → fuse with enc1 (32ch) → 32ch output
    Stage 1: operates at 32ch (no upsample needed)
    """

    def __init__(self, channels=(96, 64, 32)):
        super().__init__()
        c3, c2, c1 = channels

        # Bottleneck → decoder stage 3 processing (at 64x64)
        self.dec3 = EncoderStage(c3, num_blocks=1)  # Process bottleneck output

        # Upsample 64x64 → 128x128
        self.up3 = LightweightUpsample(c3, c2)
        # Fuse with enc2 skip (64ch) → 64ch
        self.dec2 = DecoderStage(in_channels=c2, skip_channels=c2, out_channels=c2, num_blocks=2)

        # Upsample 128x128 → 256x256
        self.up2 = LightweightUpsample(c2, c1)
        # Fuse with enc1 skip (32ch) → 32ch
        self.dec1 = DecoderStage(in_channels=c1, skip_channels=c1, out_channels=c1, num_blocks=2)

    def forward(self, enc1, enc2, bottleneck):
        # Decoder stage 3 at bottleneck resolution
        x = self.dec3(bottleneck)

        # Upsample + skip from encoder stage 2
        x = self.up3(x)
        x = self.dec2(x, enc2)

        # Upsample + skip from encoder stage 1
        x = self.up2(x)
        x = self.dec1(x, enc1)

        return x

---
## 10. Complete Dehazing Model

Assembles: Input Projection → Encoder → Multi-Scale Bottleneck → Decoder → Output Projection.

Uses global residual learning: `output = sigmoid(residual + input)` to ease optimization.

In [ ]:
class LightweightDehazingNet(nn.Module):
    """
    Lightweight Spatially Adaptive Dehazing Network.

    Architecture:
        Input (B,3,256,256)
          ↓
        Input Projection (3 → 32)
          ↓
        Encoder: Stage1(32) → Down → Stage2(64) → Down → Stage3(96)
          ↓
        Multi-Scale Bottleneck (96)
          ↓
        Decoder: Stage3(96→64) → Up+Skip → Stage2(64→32) → Up+Skip → Stage1(32)
          ↓
        Output Projection (32 → 3)
          ↓
        Global Residual: sigmoid(output + input)
          ↓
        Dehazed image (B,3,256,256) in [0,1]

    Global residual learning strategy:
        The network learns a residual correction. The final output is
        sigmoid(residual + input), ensuring the output stays in [0,1].
    """

    def __init__(self, in_channels=3, base_channels=32):
        super().__init__()
        c1 = base_channels       # 32
        c2 = base_channels * 2   # 64
        c3 = base_channels * 3   # 96

        # Input projection: 3 → 32
        self.input_proj = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, c1),
            nn.GELU()
        )

        # Encoder
        self.encoder = Encoder(channels=(c1, c2, c3))

        # Multi-scale bottleneck
        self.bottleneck = MultiScaleBottleneck(c3)

        # Decoder
        self.decoder = Decoder(channels=(c3, c2, c1))

        # Output projection: 32 → 3
        self.output_proj = nn.Conv2d(c1, in_channels, kernel_size=3, padding=1, bias=True)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Kaiming initialization for conv layers."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Store input for global residual
        identity = x  # (B, 3, H, W)

        # Input projection
        x = self.input_proj(x)  # (B, 32, 256, 256)

        # Encoder
        enc1, enc2, enc3 = self.encoder(x)
        # enc1: (B, 32, 256, 256)
        # enc2: (B, 64, 128, 128)
        # enc3: (B, 96, 64, 64)

        # Bottleneck
        bottleneck = self.bottleneck(enc3)  # (B, 96, 64, 64)

        # Decoder
        decoded = self.decoder(enc1, enc2, bottleneck)  # (B, 32, 256, 256)

        # Output projection
        residual = self.output_proj(decoded)  # (B, 3, 256, 256)

        # Global residual learning: output = sigmoid(residual + input)
        output = torch.sigmoid(residual + identity)

        return output

---
## 11. Loss Functions

Combined loss: Charbonnier (smooth L1) + SSIM.
- Charbonnier handles pixel-level reconstruction
- SSIM preserves structural similarity

In [ ]:
class CharbonnierLoss(nn.Module):
    """Charbonnier loss (smooth L1): sqrt((pred - target)^2 + eps^2)."""

    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps_sq = eps ** 2

    def forward(self, pred, target):
        diff = pred - target
        loss = torch.sqrt(diff * diff + self.eps_sq)
        return loss.mean()


class SSIMLoss(nn.Module):
    """
    SSIM loss for RGB images.
    Computes SSIM using a Gaussian window and returns 1 - SSIM.
    Implemented in pure PyTorch without external dependencies.
    """

    def __init__(self, window_size=11, channels=3):
        super().__init__()
        self.window_size = window_size
        self.channels = channels
        self.register_buffer('window', self._create_gaussian_window(window_size, channels))

    @staticmethod
    def _gaussian_kernel_1d(window_size, sigma=1.5):
        """Create 1D Gaussian kernel."""
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        return g / g.sum()

    @staticmethod
    def _create_gaussian_window(window_size, channels):
        """Create 2D Gaussian window for SSIM computation."""
        kernel_1d = SSIMLoss._gaussian_kernel_1d(window_size)
        kernel_2d = kernel_1d.unsqueeze(1) @ kernel_1d.unsqueeze(0)  # Outer product
        window = kernel_2d.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
        window = window.expand(channels, 1, window_size, window_size).contiguous()
        return window

    def _ssim(self, pred, target):
        """Compute SSIM between pred and target."""
        C1 = 0.01 ** 2
        C2 = 0.03 ** 2
        pad = self.window_size // 2

        mu_pred = F.conv2d(pred, self.window, padding=pad, groups=self.channels)
        mu_target = F.conv2d(target, self.window, padding=pad, groups=self.channels)

        mu_pred_sq = mu_pred ** 2
        mu_target_sq = mu_target ** 2
        mu_cross = mu_pred * mu_target

        sigma_pred_sq = F.conv2d(pred * pred, self.window, padding=pad, groups=self.channels) - mu_pred_sq
        sigma_target_sq = F.conv2d(target * target, self.window, padding=pad, groups=self.channels) - mu_target_sq
        sigma_cross = F.conv2d(pred * target, self.window, padding=pad, groups=self.channels) - mu_cross

        # Clamp variances for numerical stability
        sigma_pred_sq = torch.clamp(sigma_pred_sq, min=0)
        sigma_target_sq = torch.clamp(sigma_target_sq, min=0)

        ssim_map = ((2 * mu_cross + C1) * (2 * sigma_cross + C2)) / \
                   ((mu_pred_sq + mu_target_sq + C1) * (sigma_pred_sq + sigma_target_sq + C2))

        return ssim_map.mean()

    def forward(self, pred, target):
        """Returns 1 - SSIM (so lower is better, suitable as a loss)."""
        return 1.0 - self._ssim(pred, target)


class CombinedLoss(nn.Module):
    """
    Combined restoration loss:
        Loss = charbonnier_weight * Charbonnier + ssim_weight * (1 - SSIM)
    """

    def __init__(self, charbonnier_weight=0.8, ssim_weight=0.2, charbonnier_eps=1e-3):
        super().__init__()
        self.charbonnier_weight = charbonnier_weight
        self.ssim_weight = ssim_weight
        self.charbonnier = CharbonnierLoss(eps=charbonnier_eps)
        self.ssim_loss = SSIMLoss(window_size=11, channels=3)

    def forward(self, pred, target):
        loss_char = self.charbonnier(pred, target)
        loss_ssim = self.ssim_loss(pred, target)
        return self.charbonnier_weight * loss_char + self.ssim_weight * loss_ssim

---
## 12. PSNR / SSIM Metrics

Evaluation metrics computed on images in [0, 1].

In [ ]:
def calculate_psnr(pred, target, max_val=1.0):
    """
    Calculate PSNR between predicted and target images.
    Both inputs should be in [0, 1] with shape (B, C, H, W).
    Returns average PSNR across the batch.
    """
    mse = torch.mean((pred - target) ** 2, dim=[1, 2, 3])  # Per-image MSE
    # Avoid log(0) by clamping
    mse = torch.clamp(mse, min=1e-10)
    psnr = 10.0 * torch.log10((max_val ** 2) / mse)
    return psnr.mean().item()


def calculate_ssim(pred, target, window_size=11):
    """
    Calculate SSIM between predicted and target images.
    Both inputs should be in [0, 1] with shape (B, C, H, W).
    Returns average SSIM across the batch.
    """
    channels = pred.shape[1]
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    pad = window_size // 2

    # Create Gaussian window
    coords = torch.arange(window_size, dtype=torch.float32, device=pred.device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * 1.5 ** 2))
    g = g / g.sum()
    kernel_2d = g.unsqueeze(1) @ g.unsqueeze(0)
    window = kernel_2d.unsqueeze(0).unsqueeze(0).expand(channels, 1, window_size, window_size).contiguous()

    mu_pred = F.conv2d(pred, window, padding=pad, groups=channels)
    mu_target = F.conv2d(target, window, padding=pad, groups=channels)

    mu_pred_sq = mu_pred ** 2
    mu_target_sq = mu_target ** 2
    mu_cross = mu_pred * mu_target

    sigma_pred_sq = F.conv2d(pred * pred, window, padding=pad, groups=channels) - mu_pred_sq
    sigma_target_sq = F.conv2d(target * target, window, padding=pad, groups=channels) - mu_target_sq
    sigma_cross = F.conv2d(pred * target, window, padding=pad, groups=channels) - mu_cross

    sigma_pred_sq = torch.clamp(sigma_pred_sq, min=0)
    sigma_target_sq = torch.clamp(sigma_target_sq, min=0)

    ssim_map = ((2 * mu_cross + C1) * (2 * sigma_cross + C2)) / \
               ((mu_pred_sq + mu_target_sq + C1) * (sigma_pred_sq + sigma_target_sq + C2))

    # Per-image SSIM, then batch average
    ssim_per_image = ssim_map.mean(dim=[1, 2, 3])
    return ssim_per_image.mean().item()

---
## 13. Optimizer and Scheduler

In [ ]:
# Instantiate model
model = LightweightDehazingNet(in_channels=3, base_channels=32).to(DEVICE)

# Loss function
criterion = CombinedLoss(
    charbonnier_weight=CHARBONNIER_WEIGHT,
    ssim_weight=SSIM_WEIGHT,
    charbonnier_eps=CHARBONNIER_EPS
).to(DEVICE)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

# Mixed precision scaler
scaler = GradScaler('cuda')

print("Optimizer: AdamW")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})")

---
## 14. Checkpoint Utilities

In [ ]:
def save_checkpoint(state, filepath):
    """Save checkpoint to disk."""
    torch.save(state, filepath)


def load_checkpoint(filepath, model, optimizer, scheduler, scaler):
    """
    Load checkpoint and restore all training state.
    Returns (start_epoch, best_psnr) or (0, 0.0) if loading fails.
    """
    if not os.path.exists(filepath):
        print(f"No checkpoint found at {filepath}")
        return 0, 0.0

    try:
        checkpoint = torch.load(filepath, map_location=DEVICE, weights_only=False)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

        start_epoch = checkpoint['epoch'] + 1  # Resume from next epoch
        best_psnr = checkpoint.get('best_psnr', 0.0)

        print(f"Checkpoint loaded from {filepath}")
        print(f"  Resuming from epoch {start_epoch}")
        print(f"  Best PSNR so far: {best_psnr:.2f} dB")
        print(f"  Training loss: {checkpoint.get('training_loss', 'N/A')}")
        print(f"  Validation loss: {checkpoint.get('validation_loss', 'N/A')}")
        print(f"  Validation PSNR: {checkpoint.get('validation_psnr', 'N/A')}")
        print(f"  Validation SSIM: {checkpoint.get('validation_ssim', 'N/A')}")

        return start_epoch, best_psnr

    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Starting from scratch.")
        return 0, 0.0


# Try to resume training
start_epoch = 0
best_psnr = 0.0

if RESUME_TRAINING:
    latest_ckpt = os.path.join(CHECKPOINT_DIR, "latest.pth")
    start_epoch, best_psnr = load_checkpoint(
        latest_ckpt, model, optimizer, scheduler, scaler
    )

print(f"\nStarting training from epoch {start_epoch}")
print(f"Best PSNR: {best_psnr:.2f} dB")

---
## 15. Model Complexity

Parameter count and architecture overview.

In [ ]:
def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def estimate_flops(model, input_size=(1, 3, 256, 256)):
    """
    Estimate FLOPs using a simple forward hook counter.
    Counts multiply-accumulate operations (MACs) for Conv2d and Linear layers.
    FLOPs ≈ 2 × MACs.
    """
    macs = 0
    hooks = []

    def conv_hook(module, inp, out):
        nonlocal macs
        batch_size = inp[0].shape[0]
        output_dims = out.shape[2:]  # H, W
        kernel_ops = module.weight.shape[1] * module.weight.shape[2] * module.weight.shape[3]
        output_elements = out.shape[1] * output_dims[0] * output_dims[1]
        macs += kernel_ops * output_elements * batch_size

    def linear_hook(module, inp, out):
        nonlocal macs
        batch_size = inp[0].shape[0]
        macs += module.in_features * module.out_features * batch_size

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))

    device = next(model.parameters()).device
    dummy = torch.randn(input_size, device=device)
    model.eval()
    with torch.no_grad():
        model(dummy)

    for h in hooks:
        h.remove()

    flops = 2 * macs  # FLOPs ≈ 2 × MACs
    return flops


# Print model info
total_params, trainable_params = count_parameters(model)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Parameter size:       {total_params * 4 / (1024**2):.2f} MB (FP32)")
print()

try:
    flops = estimate_flops(model)
    print(f"Estimated FLOPs:      {flops / 1e9:.2f} GFLOPs")
    print(f"Estimated MACs:       {flops / 2 / 1e9:.2f} GMACs")
except Exception as e:
    print(f"FLOPs estimation failed: {e}")

print()
print("=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print(model)

---
## 16. Sanity Check

Verify model I/O shapes and data pipeline. **No training here.**

In [ ]:
# Sanity check: one batch forward pass (no weight updates)
print("Running sanity check...")
print()

model.eval()
with torch.no_grad():
    # Get one batch
    sample_hazy, sample_clean = next(iter(test_loader))
    sample_hazy = sample_hazy.to(DEVICE, non_blocking=True)
    sample_clean = sample_clean.to(DEVICE, non_blocking=True)

    # Forward pass
    sample_output = model(sample_hazy)

    print(f"Input shape:   {sample_hazy.shape}")
    print(f"Output shape:  {sample_output.shape}")
    print(f"Target shape:  {sample_clean.shape}")
    print()
    print(f"Input range:   [{sample_hazy.min().item():.4f}, {sample_hazy.max().item():.4f}]")
    print(f"Output range:  [{sample_output.min().item():.4f}, {sample_output.max().item():.4f}]")
    print(f"Target range:  [{sample_clean.min().item():.4f}, {sample_clean.max().item():.4f}]")
    print()

    # Verify shapes match
    assert sample_hazy.shape == sample_output.shape, "Shape mismatch!"
    assert sample_output.shape == sample_clean.shape, "Shape mismatch!"
    print("✓ Shape check passed: Input, Output, Target shapes match.")

    # Verify output range
    assert sample_output.min() >= 0.0 and sample_output.max() <= 1.0, "Output out of [0,1]!"
    print("✓ Range check passed: Output is in [0, 1].")

    # Test loss computation
    test_loss = criterion(sample_output, sample_clean)
    print(f"✓ Loss computation: {test_loss.item():.4f}")

    # Test metrics
    test_psnr = calculate_psnr(sample_output, sample_clean)
    test_ssim = calculate_ssim(sample_output, sample_clean)
    print(f"✓ PSNR: {test_psnr:.2f} dB")
    print(f"✓ SSIM: {test_ssim:.4f}")

    # Parameter count
    total_p, train_p = count_parameters(model)
    print(f"✓ Parameters: {total_p:,} total, {train_p:,} trainable")

print()
print("Sanity check passed! Ready to train.")

# Clean up
del sample_hazy, sample_clean, sample_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 17. Training Loop

Main training loop with:
- Mixed precision (AMP)
- Per-epoch validation
- Checkpoint saving (latest + best)
- Metric tracking

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch. Returns average training loss."""
    model.train()
    running_loss = 0.0
    num_batches = 0

    for hazy, clean in loader:
        hazy = hazy.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward
        with autocast('cuda'):
            output = model(hazy)
            loss = criterion(output, clean)

        # Scaled backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        num_batches += 1

    return running_loss / num_batches

---
## 18. Validation

In [ ]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    """
    Run validation. Returns (avg_loss, avg_psnr, avg_ssim).
    Metrics are computed per-image and averaged across the dataset.
    """
    model.eval()
    running_loss = 0.0
    running_psnr = 0.0
    running_ssim = 0.0
    num_batches = 0

    for hazy, clean in loader:
        hazy = hazy.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)

        with autocast('cuda'):
            output = model(hazy)
            loss = criterion(output, clean)

        # Compute output in float32 for accurate metrics
        output_fp32 = output.float()
        clean_fp32 = clean.float()

        running_loss += loss.item()
        running_psnr += calculate_psnr(output_fp32, clean_fp32)
        running_ssim += calculate_ssim(output_fp32, clean_fp32)
        num_batches += 1

    avg_loss = running_loss / num_batches
    avg_psnr = running_psnr / num_batches
    avg_ssim = running_ssim / num_batches

    return avg_loss, avg_psnr, avg_ssim

In [ ]:
# ==================== MAIN TRAINING LOOP ====================

# History tracking
train_losses = []
val_losses = []
psnr_history = []
ssim_history = []
lr_history = []

print("=" * 70)
print("TRAINING START")
print(f"Epochs: {start_epoch} → {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print("=" * 70)
print()

total_train_time = 0.0

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    # --- Train ---
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)

    # --- Validate ---
    val_loss, val_psnr, val_ssim = validate(model, test_loader, criterion, DEVICE)

    # --- Scheduler step ---
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # --- Track history ---
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    psnr_history.append(val_psnr)
    ssim_history.append(val_ssim)
    lr_history.append(current_lr)

    epoch_time = (time.time() - epoch_start) / 60.0
    total_train_time += epoch_time

    # --- Print epoch summary ---
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  PSNR:       {val_psnr:.2f} dB")
    print(f"  SSIM:       {val_ssim:.4f}")
    print(f"  LR:         {current_lr:.6f}")
    print(f"  Time:       {epoch_time:.2f} min")

    # --- Save latest checkpoint ---
    checkpoint_state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_psnr': best_psnr,
        'training_loss': train_loss,
        'validation_loss': val_loss,
        'validation_psnr': val_psnr,
        'validation_ssim': val_ssim
    }
    save_checkpoint(checkpoint_state, os.path.join(CHECKPOINT_DIR, "latest.pth"))

    # --- Save best model ---
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        checkpoint_state['best_psnr'] = best_psnr
        save_checkpoint(checkpoint_state, os.path.join(CHECKPOINT_DIR, "best_model.pth"))
        print(f"  ★ New best PSNR: {best_psnr:.2f} dB → saved best_model.pth")

    # --- Save periodic checkpoint every 10 epochs ---
    if (epoch + 1) % 10 == 0:
        periodic_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch + 1}.pth")
        save_checkpoint(checkpoint_state, periodic_path)
        print(f"  Saved periodic checkpoint: epoch_{epoch + 1}.pth")

    print()

print("=" * 70)
print("TRAINING COMPLETE")
print(f"Total training time: {total_train_time:.2f} min")
print(f"Best PSNR: {best_psnr:.2f} dB")
print("=" * 70)

---
## 19. Training Curves

In [ ]:
def plot_training_curves(train_losses, val_losses, psnr_history, ssim_history, lr_history):
    """Plot training metrics: loss, PSNR, SSIM, and learning rate."""
    epochs_range = range(1, len(train_losses) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Training Curves - Lightweight Dehazing Network', fontsize=14, fontweight='bold')

    # Loss
    axes[0, 0].plot(epochs_range, train_losses, 'b-', label='Train Loss', linewidth=1.5)
    axes[0, 0].plot(epochs_range, val_losses, 'r-', label='Val Loss', linewidth=1.5)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training vs Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # PSNR
    axes[0, 1].plot(epochs_range, psnr_history, 'g-', linewidth=1.5)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('PSNR (dB)')
    axes[0, 1].set_title('PSNR vs Epoch')
    axes[0, 1].grid(True, alpha=0.3)

    # SSIM
    axes[1, 0].plot(epochs_range, ssim_history, 'm-', linewidth=1.5)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('SSIM')
    axes[1, 0].set_title('SSIM vs Epoch')
    axes[1, 0].grid(True, alpha=0.3)

    # Learning Rate
    axes[1, 1].plot(epochs_range, lr_history, 'c-', linewidth=1.5)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate vs Epoch')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].ticklabel_format(axis='y', style='scientific', scilimits=(-4, -4))

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Training curves saved to /kaggle/working/training_curves.png")


# Plot if training history is available
if len(train_losses) > 0:
    plot_training_curves(train_losses, val_losses, psnr_history, ssim_history, lr_history)
else:
    print("No training history to plot.")

---
## 20. Visualization

Display side-by-side comparisons: Hazy Input | Predicted Dehazed | Ground Truth

In [ ]:
def visualize_results(model, loader, device, num_images=4):
    """
    Visualize dehazing results on test images.
    Shows: Hazy Input | Predicted Dehazed | Ground Truth
    """
    model.eval()

    # Get a batch from the loader
    hazy_batch, clean_batch = next(iter(loader))
    hazy_batch = hazy_batch.to(device, non_blocking=True)

    with torch.no_grad():
        with autocast('cuda'):
            output_batch = model(hazy_batch)

    # Move to CPU and convert to numpy
    hazy_np = hazy_batch.cpu().numpy()
    output_np = output_batch.float().cpu().numpy()
    clean_np = clean_batch.numpy()

    num_show = min(num_images, hazy_np.shape[0])

    fig, axes = plt.subplots(num_show, 3, figsize=(15, 5 * num_show))
    if num_show == 1:
        axes = axes[np.newaxis, :]  # Ensure 2D indexing

    for i in range(num_show):
        # Convert from (C, H, W) to (H, W, C)
        hazy_img = np.transpose(hazy_np[i], (1, 2, 0))
        output_img = np.transpose(output_np[i], (1, 2, 0))
        clean_img = np.transpose(clean_np[i], (1, 2, 0))

        # Clip to [0, 1] for display
        hazy_img = np.clip(hazy_img, 0, 1)
        output_img = np.clip(output_img, 0, 1)
        clean_img = np.clip(clean_img, 0, 1)

        # Compute per-image PSNR
        mse_val = np.mean((output_img - clean_img) ** 2)
        if mse_val > 0:
            psnr_val = 10 * np.log10(1.0 / mse_val)
        else:
            psnr_val = float('inf')

        axes[i, 0].imshow(hazy_img)
        axes[i, 0].set_title('Hazy Input', fontsize=12)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(output_img)
        axes[i, 1].set_title(f'Dehazed (PSNR: {psnr_val:.2f} dB)', fontsize=12)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(clean_img)
        axes[i, 2].set_title('Ground Truth', fontsize=12)
        axes[i, 2].axis('off')

    plt.suptitle('Dehazing Results - Lightweight Spatially Adaptive Network',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/dehazing_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Visualization saved to /kaggle/working/dehazing_results.png")


# Load best model for visualization
best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
if os.path.exists(best_ckpt_path):
    best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])
    print(f"Loaded best model (PSNR: {best_ckpt.get('validation_psnr', 'N/A'):.2f} dB)")
else:
    print("No best model checkpoint found, using current model weights.")

visualize_results(model, test_loader, DEVICE, num_images=4)

---
## 21. Final Model Saving

In [ ]:
# Save final model
final_model_path = os.path.join(CHECKPOINT_DIR, "final_model.pth")
torch.save({
    'model_state_dict': model.state_dict(),
    'total_params': count_parameters(model)[0],
    'trainable_params': count_parameters(model)[1],
    'best_psnr': best_psnr,
    'num_epochs_trained': NUM_EPOCHS,
}, final_model_path)

print("=" * 60)
print("FINAL MODEL SUMMARY")
print("=" * 60)
print()
print(f"Best model saved at:  {os.path.join(CHECKPOINT_DIR, 'best_model.pth')}")
print(f"Final model saved at: {final_model_path}")
print(f"Latest checkpoint:    {os.path.join(CHECKPOINT_DIR, 'latest.pth')}")
print()
print(f"Best Validation PSNR: {best_psnr:.2f} dB")
print(f"Total Parameters:     {count_parameters(model)[0]:,}")
print(f"Trainable Parameters: {count_parameters(model)[1]:,}")
print()
print("Checkpoint directory contents:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    fpath = os.path.join(CHECKPOINT_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f}: {size_mb:.2f} MB")
print()
print("Done!")